# 🎯 Module 3: Logistic Regression & Classification
## Theory + Practical — Complete Guide

---

## 📚 THEORY SECTION

### Why Not Linear Regression for Classification?

Linear regression predicts continuous values. For classification (e.g., spam/not-spam), we need outputs **between 0 and 1** representing probability.

Linear regression can predict values outside [0,1], which is meaningless for probability!

### The Sigmoid Function

**Logistic Regression** uses the sigmoid (logistic) function to squash predictions to [0, 1]:

```
σ(z) = 1 / (1 + e^(-z))   where z = w₀ + w₁x₁ + ... + wₙxₙ

Properties:
  σ(z) → 0 as z → -∞
  σ(z) → 1 as z → +∞
  σ(0) = 0.5
```

**Decision rule:**
```
If σ(z) ≥ 0.5  → Class 1
If σ(z) < 0.5  → Class 0
```

### Cost Function — Binary Cross-Entropy (Log Loss)

```
L(w) = -(1/n) Σ [yᵢ·log(ŷᵢ) + (1-yᵢ)·log(1-ŷᵢ)]

Intuition:
  When y=1 and ŷ→1: loss → 0  (correct, no penalty)
  When y=1 and ŷ→0: loss → ∞  (wrong, heavy penalty)
  When y=0 and ŷ→0: loss → 0  (correct, no penalty)
  When y=0 and ŷ→1: loss → ∞  (wrong, heavy penalty)
```

### Multi-Class Classification

| Strategy | Description |
|----------|-------------|
| **OvR (One-vs-Rest)** | Train K binary classifiers, pick highest probability |
| **OvO (One-vs-One)** | Train K(K-1)/2 classifiers, vote |
| **Softmax** | Single model with K output probabilities summing to 1 |

### Classification Metrics

```
Confusion Matrix:
                 Predicted Positive  Predicted Negative
Actual Positive       TP                  FN
Actual Negative       FP                  TN

Accuracy  = (TP + TN) / (TP + TN + FP + FN)
Precision = TP / (TP + FP)   ← Of all predicted positive, how many are actually positive?
Recall    = TP / (TP + FN)   ← Of all actual positive, how many did we catch?
F1        = 2 × (Precision × Recall) / (Precision + Recall)
```

### When to Use Precision vs Recall?

| Scenario | Metric to Optimize |
|----------|-------------------|
| Cancer detection | **Recall** (don't miss sick patients) |
| Email spam filter | **Precision** (don't delete important emails) |
| Fraud detection | **Recall** (catch all fraud) |

### ROC-AUC

**ROC Curve**: Plots True Positive Rate vs False Positive Rate at various thresholds.
**AUC**: Area Under the ROC Curve (0.5 = random, 1.0 = perfect).

---

## 🔬 PRACTICAL SECTION

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer, load_iris
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)
print('✅ Libraries loaded!')

### 🔬 Practical 1: Sigmoid Function & Decision Boundary

In [ ]:
# ============================================================
# PRACTICAL 1: Sigmoid function visualization
# ============================================================

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 300)
sig = sigmoid(z)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sigmoid plot
axes[0].plot(z, sig, color='#3498DB', linewidth=3)
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.7, label='Decision threshold (0.5)')
axes[0].axvline(0, color='green', linestyle='--', alpha=0.7, label='z=0')
axes[0].fill_between(z, sig, 0.5, where=(sig > 0.5), alpha=0.1, color='green', label='Class 1')
axes[0].fill_between(z, sig, 0.5, where=(sig < 0.5), alpha=0.1, color='red', label='Class 0')
axes[0].set_title('Sigmoid Function σ(z) = 1/(1+e^(-z))', fontweight='bold')
axes[0].set_xlabel('z (linear combination of features)')
axes[0].set_ylabel('Probability')
axes[0].legend()
axes[0].text(-9, 0.9, 'Predicts Class 1\n(probability ≥ 0.5)', fontsize=10, color='green')
axes[0].text(3, 0.1, 'Predicts Class 0\n(probability < 0.5)', fontsize=10, color='red')

# Log loss visualization
p = np.linspace(0.001, 0.999, 300)
loss_y1 = -np.log(p)         # Loss when true label is 1
loss_y0 = -np.log(1 - p)     # Loss when true label is 0

axes[1].plot(p, loss_y1, 'g-', linewidth=2.5, label='Loss when y=1')
axes[1].plot(p, loss_y0, 'r-', linewidth=2.5, label='Loss when y=0')
axes[1].set_ylim(0, 7)
axes[1].set_title('Binary Cross-Entropy Loss', fontweight='bold')
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].text(0.7, 5, '← High penalty when\npredicted probability is wrong', fontsize=9)

plt.tight_layout()
plt.show()

### 🔬 Practical 2: Breast Cancer — Binary Classification

In [ ]:
# ============================================================
# PRACTICAL 2: Breast Cancer Detection (Binary Classification)
# ============================================================

# Load data
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target)  # 0=malignant, 1=benign

print('Dataset:', cancer.DESCR[:500])
print(f'\nShape: {X.shape}')
print(f'Class distribution: Malignant={sum(y==0)}, Benign={sum(y==1)}')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Train Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_s, y_train)

y_pred = lr.predict(X_test_s)
y_prob = lr.predict_proba(X_test_s)[:, 1]  # Probability of class 1 (benign)

print(f'\nAccuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall: {recall_score(y_test, y_pred):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')

In [ ]:
# Comprehensive evaluation visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Logistic Regression — Breast Cancer Classification', fontsize=14, fontweight='bold')

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
            xticklabels=['Malignant', 'Benign'], yticklabels=['Malignant', 'Benign'])
axes[0, 0].set_title('Confusion Matrix')
axes[0, 0].set_ylabel('True Label')
axes[0, 0].set_xlabel('Predicted Label')
tn, fp, fn, tp = cm.ravel()
axes[0, 0].text(0.5, -0.3, f'TP={tp} TN={tn} FP={fp} FN={fn}', 
                 transform=axes[0, 0].transAxes, ha='center', fontsize=10)

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[0, 1].plot(fpr, tpr, color='#3498DB', linewidth=2.5, label=f'ROC Curve (AUC = {auc:.3f})')
axes[0, 1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUC = 0.5)')
axes[0, 1].fill_between(fpr, tpr, alpha=0.1, color='blue')
axes[0, 1].set_title('ROC Curve')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate (Recall)')
axes[0, 1].legend()

# Precision-Recall Curve
precision_arr, recall_arr, _ = precision_recall_curve(y_test, y_prob)
avg_prec = average_precision_score(y_test, y_prob)
axes[1, 0].plot(recall_arr, precision_arr, color='#E74C3C', linewidth=2.5,
                label=f'PR Curve (AP = {avg_prec:.3f})')
axes[1, 0].set_title('Precision-Recall Curve')
axes[1, 0].set_xlabel('Recall')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()

# Probability Distribution
axes[1, 1].hist(y_prob[y_test == 0], bins=25, alpha=0.7, color='#E74C3C', label='Malignant (True)')
axes[1, 1].hist(y_prob[y_test == 1], bins=25, alpha=0.7, color='#2ECC71', label='Benign (True)')
axes[1, 1].axvline(0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold')
axes[1, 1].set_title('Predicted Probability Distribution')
axes[1, 1].set_xlabel('Predicted Probability of Benign')
axes[1, 1].set_ylabel('Count')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

### 🔬 Practical 3: Threshold Tuning — Precision vs Recall Tradeoff

In [ ]:
# ============================================================
# PRACTICAL 3: Threshold Optimization
# ============================================================

thresholds = np.linspace(0.01, 0.99, 100)
precisions, recalls, f1s, accuracies = [], [], [], []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))
    accuracies.append(accuracy_score(y_test, y_pred_t))

best_f1_idx = np.argmax(f1s)
best_threshold = thresholds[best_f1_idx]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(thresholds, precisions, 'b-', linewidth=2, label='Precision')
ax.plot(thresholds, recalls, 'r-', linewidth=2, label='Recall')
ax.plot(thresholds, f1s, 'g-', linewidth=2.5, label='F1 Score')
ax.plot(thresholds, accuracies, 'm--', linewidth=1.5, label='Accuracy')
ax.axvline(best_threshold, color='black', linestyle='--', linewidth=2,
            label=f'Best Threshold={best_threshold:.2f} (F1={f1s[best_f1_idx]:.3f})')
ax.axvline(0.5, color='gray', linestyle=':', linewidth=1.5, label='Default 0.5')
ax.set_title('Precision, Recall, F1 vs Decision Threshold', fontweight='bold', fontsize=13)
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print(f'\n📌 Optimal threshold for F1: {best_threshold:.2f}')
print(f'   At default 0.5: Precision={precisions[49]:.3f}, Recall={recalls[49]:.3f}, F1={f1s[49]:.3f}')
print(f'   At optimal {best_threshold:.2f}: Precision={precisions[best_f1_idx]:.3f}, Recall={recalls[best_f1_idx]:.3f}, F1={f1s[best_f1_idx]:.3f}')

print('\n⚠️  For cancer detection, optimize RECALL (catch all cancer cases!)')
print('   For spam filter, optimize PRECISION (dont delete important emails!)')

## 🎓 Summary

| Concept | Key Takeaway |
|---------|-------------|
| Sigmoid | Squashes linear output to [0,1] probability |
| Log Loss | Penalizes confident wrong predictions heavily |
| Threshold | Default 0.5; tune based on use case |
| Precision | Of predicted positives, how many are truly positive? |
| Recall | Of actual positives, how many did we detect? |
| F1 Score | Harmonic mean of Precision and Recall |
| ROC-AUC | Overall model quality across all thresholds |

## 📖 Next: Decision Trees & Random Forests